In [ ]:
from dotenv import load_dotenv
import os
from agents import Agent, Runner, trace, function_tool, RunResult
from pydantic import BaseModel, Field
from pypdf import PdfReader
import json
from pathlib import Path
from datetime import date
from decimal import Decimal

load_dotenv(override=True)
NANO = os.getenv("NANO")
MINI = os.getenv("MINI")
GPT = os.getenv("GPT")

In [ ]:
class Constraint_Element(BaseModel):
    content: str = Field(description="original text")
    fulfilled: bool = Field(default=False, description="Whether the constraint is fulfilled")
    mandatory: bool = Field(default=True, description="Whether the constraint is mandatory")
    value: float|None = Field(description="The value associated with the constraint, if applicable", default=None)
    sub_constraints: list["Constraint_Element"]|None = Field(description="List of sub-constraints, if any", default=None)

class Constraint_List(BaseModel):
    constraints: list[Constraint_Element] = Field(description="List of constraint elements")


class Evidence(BaseModel):
    quote: str = Field(description="Exact supporting text from the CV")
    page: int | None = Field(default=None, description="One-based PDF page number")


class WorkExperience(BaseModel):
    employer: str | None = None
    title: str | None = None
    start_date: str | None = None
    end_date: str | None = None
    responsibilities: list[str] = Field(default_factory=list)
    evidence: list[Evidence] = Field(default_factory=list)


class Education(BaseModel):
    institution: str | None = None
    qualification: str | None = None
    field: str | None = None
    start_date: str | None = None
    end_date: str | None = None
    evidence: list[Evidence] = Field(default_factory=list)

class Project(BaseModel):
    name: str
    description: str | None = None
    technologies: list[str] = Field(default_factory=list)
    evidence: list[Evidence] = Field(default_factory=list)

class CompetencyArea(BaseModel):
    area: str
    summary: str | None = None
    skills: list[str] = Field(default_factory=list)
    evidence: list[Evidence] = Field(default_factory=list)

class Language(BaseModel):
    name: str
    proficiency: str | None = None

class CVProfile(BaseModel):
    name: str | None = None
    summary: str | None = None
    work_experience: list[WorkExperience] = Field(default_factory=list)
    education: list[Education] = Field(default_factory=list)
    projects: list[Project] = Field(default_factory=list)
    professional_competencies: list[CompetencyArea] = Field(default_factory=list)
    programming_languages: list[str] = Field(default_factory=list)
    languages: list[Language] = Field(default_factory=list)
    certifications: list[str] = Field(default_factory=list)


class ContractParty(BaseModel):
    role: str  # employer or employee
    name: str
    address: str | None = None
    representative: str | None = None


class ContractFact(BaseModel):
    key: str  # stable name, e.g. "annual_target_gross"
    value: str
    unit: str | None = None
    condition: str | None = None
    source_quote: str | None = None


class ContractSection(BaseModel):
    number: int
    title: str
    summary: str
    facts: list[ContractFact] = Field(default_factory=list)


class Contract(BaseModel):
    title: str
    version: str | None = None
    contract_date: date | None = None
    parties: list[ContractParty] = Field(default_factory=list)
    sections: list[ContractSection] = Field(default_factory=list)
   


In [ ]:
@function_tool
def read_txt() -> str:
    with open("constraints.txt", "r") as file:
        return file.read()

@function_tool
def read_pdf(filename: str) -> str:
    reader = PdfReader(filename)
    contents = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            contents += text
    #print(contents)
    return contents

def _save_json(data: BaseModel, filename: str) -> str:
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(data.model_dump_json(indent=4), encoding="utf-8")
    return f"Saved {filename}"

@function_tool
def save_constraints(data: Constraint_List, filename: str) -> str:
    """Save the generated constraints as JSON."""
    return _save_json(data, filename)


@function_tool
def save_cv(data: CVProfile, filename: str) -> str:
    """Save the extracted CV profile as JSON."""
    return _save_json(data, filename)

@function_tool
def save_contract(data: Contract, filename: str) -> str:
    """Save the extracted Contract as JSON."""
    return _save_json(data, filename)

In [ ]:
instructions = """
using provided tools, read constraints from file "constraints.txt".
some of constraints might be nested within sub-constraints. in such cases, ensure to recursively print all sub-constraints.
once result is obtained, write generated constraints to a JSON file "output/constraints.json" using tool "save_constraints".
"""
constraints_agent = Agent(name="reader of constraints", instructions=instructions, model=NANO, tools=[read_txt, save_constraints], output_type=Constraint_List )
with trace("constraints"):
    result = await Runner.run(constraints_agent, "read constraints and print its content")


In [ ]:
instructions_contract = """
using provided tools, read the content of the contract.
use file "contract.pdf".
save the extracted contract content into the Contract structure and to "output/contract.json".
"""
contract_agent = Agent(name="reader of Contract", instructions=instructions_contract, model=MINI, tools=[read_pdf, save_contract], output_type=Contract)
with trace("contract"):
    contract_result = await Runner.run(contract_agent, "Read the contract and extract its contents into the Contract structure.")
    
    #print(contract_result.final_output)  
    print(contract_result.final_output.model_dump_json(indent=4))

In [ ]:
instructions_cv = """
using provided tools, read the content of the CV.
use file "cv.pdf".
once result is obtained, write generated CV content to a JSON file "output/cv.json".
"""
cv_agent = Agent(name="reader of CV", instructions=instructions_cv, model=MINI, tools=[read_pdf, save_cv], output_type=CVProfile)
with trace("cv"):
    cv_result = await Runner.run(cv_agent, "read the content of the CV and print it")
    print(cv_result.final_output.model_dump_json(indent=4))

In [ ]:
print(len(result.final_output.constraints))
print(result.final_output.constraints[20])

In [ ]:
with open("output/contract.json", encoding="utf-8") as file:
    contract_json = json.load(file)
with open("output/cv.json", encoding="utf-8") as file:
    cv_json = json.load(file)

contract_str = json.dumps(contract_json, indent=4, ensure_ascii=False)
cv_str = json.dumps(cv_json, indent=4, ensure_ascii=False)

compare = f""""
you are the an expert job interviewer, hired by job applicant.
your task is to compare the proposed contract and the CV and provide an analysis from the point of view of job applicant.
is the contract aligned with the CV? 

contract: 
{contract_str}

cv: 
{cv_str}

answer in md format.
- which aspects of the contract are aligned with the CV.
- which aspects of the contract are misaligned with the CV.
- which aspects are to be expected from the contract due to CV.
- create Bewerber-Risikoanalyse mit Prioritätenliste
- create eine Fragenliste für das Vertragsgespräch.
write your analysis in a clear and concise manner
"""

matchmaker_agent = Agent(name="job interviewer", instructions=compare, model=GPT)
with trace("matchmaker_agent1"):
    matchmaker_result = await Runner.run(matchmaker_agent, "compare the contract and the CV and provide an analysis")
    #print(matchmaker_result.final_output)

In [ ]:
matchmaker_result.final_output
output_path = Path("output/matchmaker.md")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(
    str(matchmaker_result.final_output),
    encoding="utf-8",
)

In [ ]:
with open("output/contract.json", encoding="utf-8") as file:
    contract_json = json.load(file)
with open("output/constraints.json", encoding="utf-8") as file:
    constraints_json = json.load(file)

contract_str = json.dumps(contract_json, indent=4, ensure_ascii=False)
constraints_str = json.dumps(constraints_json, indent=4, ensure_ascii=False)

compare = f""""
you are the an expert job interviewer, hired by job applicant.
your task is to compare the proposed contract and the constraints and provide an analysis from the point of view of job applicant.
is the contract aligned with the constraints? 

contract: 
{contract_str}

constraints: 
{constraints_str}

answer in md format.
- which aspects of the contract are aligned with the constraints.
- which aspects of the contract are misaligned with the constraints.
- which aspects are to be expected from the contract due to the constraints.
- create Bewerber-Risikoanalyse mit Prioritätenliste
- create eine Fragenliste für das Vertragsgespräch.
write your analysis in a clear and concise manner
"""

matchmaker_agent = Agent(name="job interviewer", instructions=compare, model=GPT)
with trace("matchmaker_agent_constraints"):
    matchmaker_result = await Runner.run(matchmaker_agent, "compare the contract and the constraints and provide an analysis")
    #print(matchmaker_result.final_output)

In [ ]:
matchmaker_result.final_output
output_path = Path("output/constraints.md")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(
    str(matchmaker_result.final_output),
    encoding="utf-8",
)